# NammaTwin on Colab

Runs the GPU stages (draft poses, training) on a free T4. Set **Runtime > Change runtime type > T4 GPU** first.

This notebook was written from the upstream install notes and has not been executed end to end by its author (no GPU was available). If the setup cell fails, read the comments in `scripts/setup_gpu.sh`: each tool's own install page is linked there.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Replace with your own fork once you have pushed it
REPO = 'https://github.com/YOUR_USER/namma-twin.git'
!git clone --depth 1 $REPO namma-twin
%cd namma-twin

## 1. Install (about 10 minutes, once per session)

In [ ]:
!bash scripts/setup_gpu.sh

## 2. Upload the walkthrough video

Film it by following `docs/SOP.md`. Large files upload faster from Google Drive: mount Drive and copy instead of using the upload button.

In [ ]:
from google.colab import files
up = files.upload()
VIDEO = next(iter(up))
print(VIDEO)

## 3. First pass: lint, draft, align, audit

Set `MARKER_SIZE` to the edge of the printed black square in metres, as measured with a ruler. Delete it if you used no marker.

In [ ]:
import os
os.environ['MARKER_SIZE'] = '0.150'
os.environ['TITLE'] = 'Our space'
!bash scripts/run_pipeline.sh "$VIDEO" work

In [ ]:
from IPython.display import Image, HTML, display
display(Image('work/lint_timeline_c00.png'))
display(Image('work/audit/recapture_ticket.png'))
display(HTML(open('work/audit/recapture_ticket.html').read().replace("<img alt='coverage map' src='recapture_ticket.png'>", '')))

## 4. Top up (repeat until the audit says READY)

Shoot the ticket's items as one short clip, starting from somewhere the first video already covered. Upload it and run this cell.

In [ ]:
up = files.upload()
CLIP = next(iter(up))
!bash scripts/run_pipeline.sh --append "$CLIP" work
display(Image('work/audit/recapture_ticket.png'))

## 5. Finish: final poses, training, packaging

About 30 to 50 minutes on a T4 for a flat. If COLMAP registers too few keyframes, set `FINAL_POSES` to `draft` and run again.

In [ ]:
os.environ['FINAL_POSES'] = 'colmap'      # or 'draft'
os.environ['MAX_GAUSSIANS'] = '1500000'   # lower this for a faster, lighter twin
!bash scripts/run_pipeline.sh --finish work

## 6. Build the website and download it

Unzip and host the folder on GitHub Pages, Netlify or any static host.

In [ ]:
!cd viewer && npm install --silent && npm run build
!cat viewer/public/scene/report.json
!cd viewer && zip -qr ../viewer_dist.zip dist
files.download('viewer_dist.zip')

Keep the `work` folder too (copy it to Drive): the lint report, both tickets and `report.json` are the evidence for the technical abstract.